# Model Visualization

The most common way to visualize model output is through the use of plots. In this section, we will create a series of plots to visualize the output from our model runs. We will focus on the following:

- 2D plot in python using `ModVis` package
- 3D plot in ParaView/VisIt

## ModVis

`ModVis` is a python package for visualizing model output in 2D. It allows us to create a variety of plots including surface and subsurface variables to understand the behavior of our model. The documentation for `ModVis` can be found [here](https://pinshuai.github.io/modvis/).


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Import Relevant Packages
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

import modvis.ats_xdmf as ats_xdmf
import modvis.plot_vis_file as pv
import modvis.ATSutils as ATSutils
import modvis.utils as utils
import modvis.general_plots as gp

In [ ]:
# ats output directory
dir = './vis_files/'

### Loading data

In [ ]:
vis_surface = ats_xdmf.VisFile(dir, domain='surface', ats_version=1.4, 
                               model_time_unit = 'd',
                               mixed_element=True)
vis_subsurface = ats_xdmf.VisFile(dir, ats_version=1.4, 
                               model_time_unit = 'd',
                               mixed_element=True)

### Plot Surface Field

In [ ]:
fig, ax, gdf1 = pv.plot_surface_data(vis_surface, var_name="surface-ponded_depth", 
                                    time_slice=16, robust=True,
                                    mixed_element=True)

Add `data_lim` parameter to set the threshold for data visualization. 


In [ ]:
fig, ax, gdf2 = pv.plot_surface_data(vis_surface, var_name="surface-ponded_depth", 
                                    data_lim=[0.01, None],
                                    time_slice=16, robust=True,
                                    mixed_element=True)

### Plot Subsurface Field

Note, layer index starts from 0 at the top layer.

In [ ]:
fig, ax, gdf1= pv.plot_layer_data(vis_subsurface, var_name="saturation_liquid", 
                            time_slice=16, layer_ind = 0, 
                            robust=True,
                            mixed_element=True)

### Model evaluation

There is no streamflow data available at the Coweeta watershed outlet. Instead, the streamflow is measured at several subcatchments. 

Here we just show an example of how you can perform model evaluation at the CoalCreek Watershed in Colorado. 

We can show water balance plot for all subdomains including canopy, snow, surface, and subsurface or the entire "global" domain. It is important to check if max error is close to zero! Otherwise, there may be a water balance issue in the model.

In [ ]:
simu_df = ATSutils.load_waterBalance(dir, WB_filename="water_balance-daily.dat", 
                                     domain_names=['global'], catchment_area=53159325,
                                plot = True)

- Load observation data

Load USGS gage data by providing the gage ID (e.g., "09111250").

In [ ]:
obs_df = utils.load_nwis(sites= "09111250", start = '2014-10-01')
obs_df.plot()

- Streamflow comparison

Compare simulated streamflow with observed USGS streamflow. Note the large peak flow at the begining or the simulation is due to spinups. Suggest to discard the first year or two for calibration.

In [ ]:
fig,ax = plt.subplots(1,1, figsize=(8,4), dpi=150)
simu_df['watershed boundary discharge [m^3/d]'].plot(color = 'c',ax=ax, label= "simu_Q")
obs_df['Discharge [m^3/d]'].plot(color = 'k', ax=ax, label = "obs_Q")
ax.set_ylabel("Discharge [m^3/d]")
ax.legend()

- FDC comparison
  
Compare the simulated vs. observed flow duration curve. This is helpful for determine if the model is under/over-estimate high/low flows.

In [ ]:
fig, ax = gp.plot_FDC(dfs=[obs_df['Discharge [m^3/d]'], simu_df['watershed boundary discharge [m^3/d]']],
           labels=['obs_Q','simu_Q'], 
           colors=['k', 'c'],
           start_date="2016-10-01" 
           )

- One-to-one plot

One to one scatter plot with common metrics such as R^2, NSE, and mKGE (modified KGE). The closer the metric is to one, the better the simulation.

In [ ]:
gp.one2one_plot(obs_df['Discharge [m^3/d]'], simu_df['watershed boundary discharge [m^3/d]'],
               metrics=['R^2', 'NSE', 'mKGE'],
                # metrics='all',
               show_density=False,
                start_date="2016-10-01"
               )

## ParaView

Open ParaView and load the simulation output files (e.g., .`ats_vis_data.VisIt.xmf` files) to visualize the results.
